In [1]:
import os
import glob
import random
import pandas as pd
from tqdm.auto import tqdm


tqdm.pandas()

In [2]:
# ============================================================
# Configuration
# ============================================================

# Folder containing all instruction CSV files
DATASET_FOLDER = "/kaggle/input/datasets/punitkashyap2007/virgo-instruction"

# Output folder
OUTPUT_FOLDER = "/kaggle/working/virgo_chat_dataset"

os.makedirs(OUTPUT_FOLDER, exist_ok=True)

# Train/Validation split
TRAIN_RATIO = 0.98
RANDOM_SEED = 42

random.seed(RANDOM_SEED)

In [3]:
# ============================================================
# Load all CSV files (Robust Version)
# ============================================================

import csv

csv_files = sorted(glob.glob(os.path.join(DATASET_FOLDER, "*.csv")))

datasets = []

for file in csv_files:

    print(f"Loading {os.path.basename(file)}")

    try:
        df = pd.read_csv(
            file,
            engine="python",
            quoting=csv.QUOTE_MINIMAL,
            on_bad_lines="skip"
        )

        # lowercase columns
        df.columns = [c.strip().lower() for c in df.columns]

        # keep only first two columns
        df = df.iloc[:, :2]

        df.columns = ["prompt", "response"]

        df = df.dropna()

        df["prompt"] = df["prompt"].astype(str).str.strip()
        df["response"] = df["response"].astype(str).str.strip()

        df = df[
            (df["prompt"] != "") &
            (df["response"] != "")
        ]

        datasets.append(df)

        print(f"   ✓ {len(df):,} samples")

    except Exception as e:
        print(f"   ✗ {e}")

dataset = pd.concat(datasets, ignore_index=True)

print("\n==============================")
print("Total Samples :", len(dataset))
print("==============================")

dataset.head()

Loading exact_sentence_count.csv
   ✓ 10,025 samples
Loading exact_word_count.csv
   ✓ 10,161 samples
Loading markdown.csv
   ✓ 3,999 samples
Loading number_only.csv
   ✓ 8,043 samples
Loading one_sentence_ds.csv
   ✓ 9,996 samples
Loading one_word_ds.csv
   ✓ 10,134 samples
Loading virgo_identity.csv
   ✓ 5,520 samples
Loading yes_or_no.csv
   ✓ 8,116 samples

Total Samples : 65994


,prompt,response
0,Explain the solar system in exactly 2 sentences.,The solar system consists of the Sun and the o...
1,Describe photosynthesis in exactly 3 sentences.,Photosynthesis is the process by which green p...
2,What is gravity in exactly 1 sentence?,Gravity is the force that attracts objects wit...
3,Compare mammals and reptiles in exactly 4 sent...,Mammals are warm blooded animals while reptile...
4,Summarize the history of the printing press in...,The printing press transformed the spread of k...


In [4]:
import os

for file in csv_files:
    with open(file, "r", encoding="utf-8", errors="ignore") as f:
        total_lines = sum(1 for _ in f) - 1

    df = pd.read_csv(
        file,
        engine="python",
        quoting=csv.QUOTE_MINIMAL,
        on_bad_lines="skip"
    )

    print(
        f"{os.path.basename(file):30}"
        f" Expected: {total_lines:6}"
        f" Loaded: {len(df):6}"
        f" Lost: {total_lines-len(df):6}"
    )

exact_sentence_count.csv       Expected:  10095 Loaded:  10025 Lost:     70
exact_word_count.csv           Expected:  10215 Loaded:  10162 Lost:     53
markdown.csv                   Expected:   4026 Loaded:   3999 Lost:     27
number_only.csv                Expected:   8072 Loaded:   8044 Lost:     28
one_sentence_ds.csv            Expected:  10173 Loaded:   9996 Lost:    177
one_word_ds.csv                Expected:  10150 Loaded:  10134 Lost:     16
virgo_identity.csv             Expected:   5527 Loaded:   5524 Lost:      3
yes_or_no.csv                  Expected:   8117 Loaded:   8117 Lost:      0


In [5]:
# ============================================================
# Clean Dataset
# ============================================================

print("Before cleaning :", len(dataset))

# Remove exact duplicate prompt-response pairs
dataset = dataset.drop_duplicates(
    subset=["prompt", "response"]
)

# Remove duplicate prompts (keep first response)
dataset = dataset.drop_duplicates(
    subset=["prompt"],
    keep="first"
)

# Remove leading/trailing whitespace
dataset["prompt"] = dataset["prompt"].str.strip()
dataset["response"] = dataset["response"].str.strip()

# Remove empty rows
dataset = dataset[
    (dataset["prompt"].str.len() > 0) &
    (dataset["response"].str.len() > 0)
]

# Shuffle dataset
dataset = dataset.sample(
    frac=1,
    random_state=RANDOM_SEED
).reset_index(drop=True)

print("After cleaning :", len(dataset))

dataset.head()

Before cleaning : 65994
After cleaning : 61405


,prompt,response
0,Which database is developed by Oracle?,Oracle Database.
1,Describe briefly: What is the Southern Ocean?,The Southern Ocean surrounds Antarctica and co...
2,Outline the process of fossil fuel formation i...,Ancient plants and animals accumulated over mi...
3,Will your licensing always stay the same?,Not necessarily. Virgo is currently open sourc...
4,Choose Yes or No: Is JSON an operating system?,No


In [6]:
# ============================================================
# Convert Dataset to Virgo Chat Format
# ============================================================

BOS_TOKEN = "<bos>"
EOS_TOKEN = "<eos>"
NEWLINE_TOKEN = "<newline>"

def normalize(text):
    text = str(text).strip()

    # Normalize line endings
    text = text.replace("\r\n", "\n").replace("\r", "\n")

    # Replace every newline with the special token
    text = text.replace("\n", NEWLINE_TOKEN)

    # Remove duplicate newline tokens
    while f"{NEWLINE_TOKEN}{NEWLINE_TOKEN}{NEWLINE_TOKEN}" in text:
        text = text.replace(
            f"{NEWLINE_TOKEN}{NEWLINE_TOKEN}{NEWLINE_TOKEN}",
            f"{NEWLINE_TOKEN}{NEWLINE_TOKEN}"
        )

    return text

def format_chat(prompt, response):
    prompt = normalize(prompt)
    response = normalize(response)

    return (
        f"{BOS_TOKEN}"
        f"User: {prompt}"
        f"{NEWLINE_TOKEN}{NEWLINE_TOKEN}"
        f"Assistant: {response}"
        f"{EOS_TOKEN}"
    )

dataset["text"] = dataset.apply(
    lambda row: format_chat(row["prompt"], row["response"]),
    axis=1
)

print(dataset["text"].iloc[0])

print(f"\nTotal Chat Samples: {len(dataset):,}")

<bos>User: Which database is developed by Oracle?<newline><newline>Assistant: Oracle Database.<eos>

Total Chat Samples: 61,405


In [7]:
# ============================================================
# Load Virgo Tokenizer
# ============================================================

from tokenizers import Tokenizer

TOKENIZER_PATH = "/kaggle/input/datasets/punitkashyap2007/virgo-tokenizer/virgo_tokenizer.json"

tokenizer = Tokenizer.from_file(TOKENIZER_PATH)

print("Tokenizer Loaded Successfully")
print("Vocabulary Size:", tokenizer.get_vocab_size())

Tokenizer Loaded Successfully
Vocabulary Size: 45000


In [8]:
import pickle
import pandas as pd
from sklearn.model_selection import train_test_split

# -------------------------------
# Keep only valid prompt/response pairs
# -------------------------------
df = df.copy()

df = df.dropna(subset=["prompt", "response"])

df["prompt"] = df["prompt"].astype(str).str.strip()
df["response"] = df["response"].astype(str).str.strip()

df = df[
    (df["prompt"] != "") &
    (df["response"] != "")
].reset_index(drop=True)

print(f"Total valid samples: {len(df)}")

# -------------------------------
# Split dataset
# -------------------------------
train_df, val_df = train_test_split(
    df,
    test_size=0.02,
    random_state=42,
    shuffle=True
)

print(f"Train samples : {len(train_df)}")
print(f"Validation    : {len(val_df)}")


# -------------------------------
# Convert each row into one sample
# -------------------------------
def create_samples(dataframe):

    samples = []

    for _, row in dataframe.iterrows():

        prompt = row["prompt"]
        response = row["response"]

        prompt_text = (
            "<bos>"
            "User: "
            + prompt
            + "<newline><newline>"
            + "Assistant: "
        )

        full_text = (
            prompt_text
            + response
            + "<eos>"
        )

        prompt_ids = tokenizer.encode(prompt_text).ids
        full_ids = tokenizer.encode(full_text).ids

        labels = (
            [-100] * len(prompt_ids)
            + full_ids[len(prompt_ids):]
        )

        samples.append(
            {
                "input_ids": full_ids,
                "labels": labels,
            }
        )

    return samples


train_samples = create_samples(train_df)
val_samples = create_samples(val_df)

print(f"Training Samples   : {len(train_samples)}")
print(f"Validation Samples : {len(val_samples)}")

# -------------------------------
# Save dataset
# -------------------------------
with open("train_samples.pkl", "wb") as f:
    pickle.dump(train_samples, f)

with open("val_samples.pkl", "wb") as f:
    pickle.dump(val_samples, f)

print("Saved train_samples.pkl")
print("Saved val_samples.pkl")

Total valid samples: 8116
Train samples : 7953
Validation    : 163
Training Samples   : 7953
Validation Samples : 163
Saved train_samples.pkl
Saved val_samples.pkl


In [9]:
# ============================================================
# Tokenize Dataset
# ============================================================

import numpy as np
from tqdm.auto import tqdm

OUTPUT_DIR = "/kaggle/working/virgo_chat_dataset"
os.makedirs(OUTPUT_DIR, exist_ok=True)



In [10]:
# ============================================================
# Verify SFT Dataset
# ============================================================

import pickle

with open("train_samples.pkl", "rb") as f:
    train_samples = pickle.load(f)

with open("val_samples.pkl", "rb") as f:
    val_samples = pickle.load(f)

print(f"Training Samples   : {len(train_samples):,}")
print(f"Validation Samples : {len(val_samples):,}")

sample = train_samples[0]

print("\nKeys:")
print(sample.keys())

print("\nInput Length :", len(sample["input_ids"]))
print("Label Length :", len(sample["labels"]))

assert len(sample["input_ids"]) == len(sample["labels"])

valid_tokens = sum(label != -100 for label in sample["labels"])
ignored_tokens = sum(label == -100 for label in sample["labels"])

print(f"\nAssistant Tokens : {valid_tokens}")
print(f"Ignored Tokens   : {ignored_tokens}")

max_token = max(max(s["input_ids"]) for s in train_samples)

print("\nMaximum Token ID :", max_token)
print("Vocabulary Size  :", tokenizer.get_vocab_size())

assert max_token < tokenizer.get_vocab_size(), "Found invalid token!"

print("\n✅ SFT dataset verification passed.")

Training Samples   : 7,953
Validation Samples : 163

Keys:
dict_keys(['input_ids', 'labels'])

Input Length : 18
Label Length : 18

Assistant Tokens : 1
Ignored Tokens   : 17

Maximum Token ID : 44994
Vocabulary Size  : 45000

✅ SFT dataset verification passed.


In [11]:
import pickle

with open("train_samples.pkl", "rb") as f:
    train_samples = pickle.load(f)

with open("val_samples.pkl", "rb") as f:
    val_samples = pickle.load(f)

train_tokens = sum(len(sample["input_ids"]) for sample in train_samples)
val_tokens = sum(len(sample["input_ids"]) for sample in val_samples)

dataset_info = {
    "train_samples": len(train_samples),
    "validation_samples": len(val_samples),
    "train_tokens": train_tokens,
    "validation_tokens": val_tokens,
    "vocab_size": tokenizer.get_vocab_size(),
}

print(dataset_info)

{'train_samples': 7953, 'validation_samples': 163, 'train_tokens': 186367, 'validation_tokens': 3772, 'vocab_size': 45000}


In [12]:
# ============================================================
# Decode an SFT Sample
# ============================================================

import pickle

with open("train_samples.pkl", "rb") as f:
    train_samples = pickle.load(f)

sample = train_samples[0]

print("Input IDs:")
print(sample["input_ids"][:200])

print("\nDecoded Input:\n")
print(tokenizer.decode(sample["input_ids"]))

print("\nNumber of Tokens :", len(sample["input_ids"]))
print("Assistant Tokens :", sum(l != -100 for l in sample["labels"]))
print("Ignored Tokens   :", sum(l == -100 for l in sample["labels"]))

Input IDs:
[2, 12187, 217, 31, 1443, 212, 28842, 397, 6650, 5424, 36, 4, 4, 18772, 8225, 31, 2219, 3]

Decoded Input:

User: Can a polygon have zero sides?Assistant: No

Number of Tokens : 18
Assistant Tokens : 1
Ignored Tokens   : 17


In [13]:
class CFG:

    # ==========================================================
    # Dataset
    # ==========================================================

    train_dataset = "train_samples.pkl"
    val_dataset = "val_samples.pkl"

    tokenizer_path = "/kaggle/input/datasets/punitkashyap2007/virgo-tokenizer/virgo_tokenizer.json"

    # ==========================================================
    # Checkpoint
    # ==========================================================

    checkpoint = "/kaggle/input/datasets/punitkashyap2007/virgo-chat-model/virgo_chat_best_h100_ep2.pt"

    output_dir = "/kaggle/working/virgo_chat_v2"

    # ==========================================================
    # Model (UNCHANGED)
    # ==========================================================

    vocab_size = 45000

    d_model = 768
    num_heads = 12
    num_layers = 12
    d_ff = 3072

    max_seq_length = 512

    dropout = 0.1

    # ==========================================================
    # Fine-tuning
    # ==========================================================

    epochs = 2

    batch_size = 4

    gradient_accumulation = 32

    learning_rate = 1e-5

    min_lr = 1e-6

    weight_decay = 0.01

    warmup_ratio = 0.03

    grad_clip = 1.0

    num_workers = 2

    seed = 42

In [14]:
import os

# Create output directory
os.makedirs(CFG.output_dir, exist_ok=True)

print("Output directory:", CFG.output_dir)

Output directory: /kaggle/working/virgo_chat_v2


In [15]:
# ==========================================================
# Instruction Dataset
# ==========================================================

import pickle
import torch
from torch.utils.data import Dataset


class InstructionDataset(Dataset):

    def __init__(self, filename, max_length):

        with open(filename, "rb") as f:
            self.samples = pickle.load(f)

        self.max_length = max_length

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):

        sample = self.samples[idx]

        input_ids = sample["input_ids"][:self.max_length]
        labels = sample["labels"][:self.max_length]

        original_length = len(input_ids)
        pad_length = self.max_length - original_length

        if pad_length > 0:
            # Change 0 if your tokenizer uses a different PAD token ID
            input_ids = input_ids + [0] * pad_length
            labels = labels + [-100] * pad_length

        attention_mask = [1] * original_length + [0] * pad_length

        return {
            "input_ids": torch.tensor(input_ids, dtype=torch.long),
            "labels": torch.tensor(labels, dtype=torch.long),
            "attention_mask": torch.tensor(attention_mask, dtype=torch.long),
        }

In [16]:
# ==========================================================
# Create Instruction Datasets
# ==========================================================

train_dataset = InstructionDataset(
    filename=CFG.train_dataset,
    max_length=CFG.max_seq_length
)

val_dataset = InstructionDataset(
    filename=CFG.val_dataset,
    max_length=CFG.max_seq_length
)

print("=" * 50)
print("Instruction Datasets Loaded")
print("=" * 50)
print(f"Training Samples   : {len(train_dataset):,}")
print(f"Validation Samples : {len(val_dataset):,}")

# Verify one sample
sample = train_dataset[0]

print("\nSample Shapes")
print(f"Input IDs      : {sample['input_ids'].shape}")
print(f"Labels         : {sample['labels'].shape}")
print(f"Attention Mask : {sample['attention_mask'].shape}")

print("\n✅ Dataset loaded successfully.")

Instruction Datasets Loaded
Training Samples   : 7,953
Validation Samples : 163

Sample Shapes
Input IDs      : torch.Size([512])
Labels         : torch.Size([512])
Attention Mask : torch.Size([512])

✅ Dataset loaded successfully.


In [17]:
# ==========================================================
# Create DataLoaders
# ==========================================================

from torch.utils.data import DataLoader

train_loader = DataLoader(
    dataset=train_dataset,
    batch_size=CFG.batch_size,
    shuffle=True,
    num_workers=CFG.num_workers,
    pin_memory=True,
    drop_last=True,
)

val_loader = DataLoader(
    dataset=val_dataset,
    batch_size=CFG.batch_size,
    shuffle=False,
    num_workers=CFG.num_workers,
    pin_memory=True,
    drop_last=False,
)

print("=" * 50)
print("DataLoaders Created")
print("=" * 50)
print(f"Train Batches      : {len(train_loader):,}")
print(f"Validation Batches : {len(val_loader):,}")

# ==========================================================
# Verify One Batch
# ==========================================================

batch = next(iter(train_loader))

print("\nBatch Shapes")
print(f"Input IDs      : {batch['input_ids'].shape}")
print(f"Labels         : {batch['labels'].shape}")
print(f"Attention Mask : {batch['attention_mask'].shape}")

print("\n✅ DataLoaders are working correctly.")

DataLoaders Created
Train Batches      : 1,988
Validation Batches : 41

Batch Shapes
Input IDs      : torch.Size([4, 512])
Labels         : torch.Size([4, 512])
Attention Mask : torch.Size([4, 512])

✅ DataLoaders are working correctly.


In [18]:
import torch
from tokenizers import Tokenizer

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

tokenizer = Tokenizer.from_file(CFG.tokenizer_path)

print("Vocabulary Size :", tokenizer.get_vocab_size())
print("Checkpoint Path :", CFG.checkpoint)

Device: cuda
Vocabulary Size : 45000
Checkpoint Path : /kaggle/input/datasets/punitkashyap2007/virgo-chat-model/virgo_chat_best_h100_ep2.pt


In [19]:
# ==========================================================
# Rotary Positional Embedding (RoPE)
# ==========================================================

import torch
import torch.nn as nn


class RotaryEmbedding(nn.Module):
    def __init__(self, d_k, max_seq_length=2048, base=10000.0):
        super().__init__()

        assert d_k % 2 == 0, "Head dimension must be even for RoPE"

        inv_freq = 1.0 / (base ** (torch.arange(0, d_k, 2).float() / d_k))
        positions = torch.arange(max_seq_length).float()

        freqs = torch.outer(positions, inv_freq)

        self.register_buffer(
            "cos",
            freqs.cos()[None, None, :, :],
            persistent=False,
        )

        self.register_buffer(
            "sin",
            freqs.sin()[None, None, :, :],
            persistent=False,
        )

    def forward(self, Q, K):
        seq_length = Q.size(-2)

        cos = self.cos[:, :, :seq_length, :].to(dtype=Q.dtype)
        sin = self.sin[:, :, :seq_length, :].to(dtype=Q.dtype)

        Q_even = Q[..., 0::2]
        Q_odd = Q[..., 1::2]

        K_even = K[..., 0::2]
        K_odd = K[..., 1::2]

        Q = torch.stack(
            (
                Q_even * cos - Q_odd * sin,
                Q_even * sin + Q_odd * cos,
            ),
            dim=-1,
        ).flatten(-2)

        K = torch.stack(
            (
                K_even * cos - K_odd * sin,
                K_even * sin + K_odd * cos,
            ),
            dim=-1,
        ).flatten(-2)

        return Q, K

In [20]:
class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, num_heads, max_seq_length):
        super(MultiHeadAttention, self).__init__()

        assert d_model % num_heads == 0, "d_model must be divisible by num_heads"

        self.d_model = d_model
        self.num_heads = num_heads
        self.d_k = d_model // num_heads

        self.W_q = nn.Linear(d_model, d_model)
        self.W_k = nn.Linear(d_model, d_model)
        self.W_v = nn.Linear(d_model, d_model)
        self.W_o = nn.Linear(d_model, d_model)

        self.rope = RotaryEmbedding(self.d_k, max_seq_length)

    def split_heads(self, x):
        batch_size, seq_length, _ = x.size()
        return x.view(batch_size, seq_length, self.num_heads, self.d_k).transpose(1, 2)

    def combine_heads(self, x):
        batch_size, _, seq_length, _ = x.size()
        return x.transpose(1, 2).contiguous().view(batch_size, seq_length, self.d_model)

    def forward(self, x):
        Q = self.split_heads(self.W_q(x))
        K = self.split_heads(self.W_k(x))
        V = self.split_heads(self.W_v(x))

        Q, K = self.rope(Q, K)

        attn_output = F.scaled_dot_product_attention(Q, K, V, dropout_p=0.0, is_causal=True)

        return self.W_o(self.combine_heads(attn_output))

In [21]:
class PositionWiseFeedForward(nn.Module):
    def __init__(self, d_model, d_ff):
        super(PositionWiseFeedForward, self).__init__()

        self.fc1 = nn.Linear(d_model, d_ff)
        self.fc2 = nn.Linear(d_ff, d_model)
        self.gelu = nn.GELU()

    def forward(self, x):
        return self.fc2(self.gelu(self.fc1(x)))

In [22]:
class TransformerBlock(nn.Module):
    def __init__(self, d_model, num_heads, d_ff, max_seq_length, dropout):
        super(TransformerBlock, self).__init__()

        self.self_attn = MultiHeadAttention(d_model, num_heads, max_seq_length)
        self.feed_forward = PositionWiseFeedForward(d_model, d_ff)

        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)

        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        norm_x = self.norm1(x)
        x = x + self.dropout(self.self_attn(norm_x))

        norm_x = self.norm2(x)
        x = x + self.dropout(self.feed_forward(norm_x))

        return x

In [23]:
class VirgoModel(nn.Module):
    def __init__(self, vocab_size, d_model, num_heads, num_layers, d_ff, max_seq_length, dropout):
        super(VirgoModel, self).__init__()

        self.vocab_size = vocab_size
        self.d_model = d_model
        self.max_seq_length = max_seq_length

        self.token_embedding = nn.Embedding(vocab_size, d_model)

        self.layers = nn.ModuleList([
            TransformerBlock(d_model, num_heads, d_ff, max_seq_length, dropout)
            for _ in range(num_layers)
        ])

        self.norm = nn.LayerNorm(d_model)
        self.lm_head = nn.Linear(d_model, vocab_size, bias=False)

        self.dropout = nn.Dropout(dropout)

        self.lm_head.weight = self.token_embedding.weight

    def forward(self, input_ids):
        seq_length = input_ids.size(1)

        if seq_length > self.max_seq_length:
            raise ValueError(f"Sequence length {seq_length} exceeds maximum {self.max_seq_length}")

        x = self.token_embedding(input_ids)
        x = self.dropout(x)

        for layer in self.layers:
            x = layer(x)

        x = self.norm(x)

        return self.lm_head(x)

In [24]:
# ==========================================================
# Build Model
# ==========================================================

model = VirgoModel(
    vocab_size=CFG.vocab_size,
    d_model=CFG.d_model,
    num_heads=CFG.num_heads,
    num_layers=CFG.num_layers,
    d_ff=CFG.d_ff,
    max_seq_length=CFG.max_seq_length,
    dropout=CFG.dropout,
)

model = model.to(device)

print(f"Parameters: {sum(p.numel() for p in model.parameters()):,}")

Parameters: 119,616,000


In [25]:
# ==========================================================
# Load Pretrained Checkpoint
# ==========================================================

checkpoint = torch.load(CFG.checkpoint, map_location=device)

# If the checkpoint contains extra information
if isinstance(checkpoint, dict) and "model_state_dict" in checkpoint:
    state_dict = checkpoint["model_state_dict"]
else:
    state_dict = checkpoint

missing_keys, unexpected_keys = model.load_state_dict(
    state_dict,
    strict=False
)

print("✅ Checkpoint Loaded")

print(f"Missing Keys    : {len(missing_keys)}")
print(f"Unexpected Keys : {len(unexpected_keys)}")

if len(missing_keys):
    print("\nMissing Keys:")
    for k in missing_keys:
        print(k)

if len(unexpected_keys):
    print("\nUnexpected Keys:")
    for k in unexpected_keys:
        print(k)

✅ Checkpoint Loaded
Missing Keys    : 0
Unexpected Keys : 0


In [26]:
# ==========================================================
# Optimizer & Scheduler
# ==========================================================

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=CFG.learning_rate,
    weight_decay=CFG.weight_decay
)

total_steps = len(train_loader) * CFG.epochs
warmup_steps = int(total_steps * CFG.warmup_ratio)

scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer,
    T_max=total_steps,
    eta_min=CFG.min_lr
)

criterion = nn.CrossEntropyLoss(ignore_index=-100)

print(f"Total Steps  : {total_steps}")
print(f"Warmup Steps : {warmup_steps}")
print(f"Learning Rate: {CFG.learning_rate}")

Total Steps  : 3976
Warmup Steps : 119
Learning Rate: 1e-05


In [27]:
print("Model device:", next(model.parameters()).device)

total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"Total Parameters     : {total_params:,}")
print(f"Trainable Parameters : {trainable_params:,}")

Model device: cuda:0
Total Parameters     : 119,616,000
Trainable Parameters : 119,616,000


In [28]:
# ==========================================================
# PyTorch Imports
# ==========================================================

import torch
import torch.nn as nn
import torch.nn.functional as F

In [29]:
# ==========================================================
# Sanity Check
# ==========================================================

import torch

model.train()

batch = next(iter(train_loader))

input_ids = batch["input_ids"].to(device)
labels = batch["labels"].to(device)

with torch.amp.autocast(device_type=device.type):

    logits = model(input_ids)

    loss = criterion(
        logits.reshape(-1, logits.size(-1)),
        labels.reshape(-1)
    )

print("=" * 50)
print("Sanity Check")
print("=" * 50)

print(f"Input Shape      : {input_ids.shape}")
print(f"Labels Shape     : {labels.shape}")
print(f"Logits Shape     : {logits.shape}")
print(f"Loss             : {loss.item():.6f}")

num_valid = (labels != -100).sum().item()
num_ignored = (labels == -100).sum().item()

print(f"Assistant Tokens : {num_valid}")
print(f"Ignored Tokens   : {num_ignored}")

assert logits.shape[:2] == input_ids.shape
assert labels.shape == input_ids.shape

print("\n✅ Forward pass successful.")

Sanity Check
Input Shape      : torch.Size([4, 512])
Labels Shape     : torch.Size([4, 512])
Logits Shape     : torch.Size([4, 512, 45000])
Loss             : 8.268880
Assistant Tokens : 6
Ignored Tokens   : 2042

✅ Forward pass successful.


In [30]:
# ==========================================================
# Fine-tuning Loop (SFT)
# ==========================================================

import os
import torch
from tqdm.auto import tqdm

# Create output directory if it doesn't exist
os.makedirs(CFG.output_dir, exist_ok=True)

scaler = torch.amp.GradScaler("cuda")

best_loss = float("inf")

for epoch in range(CFG.epochs):

    model.train()

    running_loss = 0.0

    optimizer.zero_grad(set_to_none=True)

    progress = tqdm(
        train_loader,
        desc=f"Epoch {epoch + 1}/{CFG.epochs}"
    )

    for step, batch in enumerate(progress):

        input_ids = batch["input_ids"].to(device, non_blocking=True)
        labels = batch["labels"].to(device, non_blocking=True)

        with torch.amp.autocast(device_type=device.type):

            logits = model(input_ids)

            loss = criterion(
                logits.reshape(-1, logits.size(-1)),
                labels.reshape(-1)
            )

            loss = loss / CFG.gradient_accumulation

        scaler.scale(loss).backward()

        if (
            (step + 1) % CFG.gradient_accumulation == 0
            or (step + 1) == len(train_loader)
        ):

            scaler.unscale_(optimizer)

            torch.nn.utils.clip_grad_norm_(
                model.parameters(),
                CFG.grad_clip,
            )

            scaler.step(optimizer)
            scaler.update()

            optimizer.zero_grad(set_to_none=True)

            scheduler.step()

        running_loss += loss.item() * CFG.gradient_accumulation

        progress.set_postfix(
            loss=f"{running_loss/(step+1):.4f}",
            lr=f"{scheduler.get_last_lr()[0]:.2e}",
        )

    epoch_loss = running_loss / len(train_loader)

    print("=" * 60)
    print(f"Epoch {epoch + 1}/{CFG.epochs}")
    print(f"Training Loss : {epoch_loss:.4f}")
    print("=" * 60)

    if epoch_loss < best_loss:

        best_loss = epoch_loss

        checkpoint = {
            "epoch": epoch + 1,
            "loss": best_loss,
            "model_state_dict": model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "scheduler_state_dict": scheduler.state_dict(),
        }

        save_path = os.path.join(
            CFG.output_dir,
            "virgo_instruction_best.pt"
        )

        torch.save(checkpoint, save_path)

        print(f"✅ Best model saved to:\n{save_path}")

print("\n🎉 Fine-tuning completed successfully!")

Epoch 1/2:   0%|          | 0/1988 [00:00<?, ?it/s]

Epoch 1/2
Training Loss : 0.3722
✅ Best model saved to:
/kaggle/working/virgo_chat_v2/virgo_instruction_best.pt


Epoch 2/2:   0%|          | 0/1988 [00:00<?, ?it/s]

Epoch 2/2
Training Loss : 0.0036
✅ Best model saved to:
/kaggle/working/virgo_chat_v2/virgo_instruction_best.pt

🎉 Fine-tuning completed successfully!


In [ ]:
# ==========================================================
# Virgo Chat Inference
# ==========================================================

import os
import time
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Using:", device)

# ==========================================================
# Load Model
# ==========================================================

model = VirgoModel(
    vocab_size=CFG.vocab_size,
    d_model=CFG.d_model,
    num_heads=CFG.num_heads,
    num_layers=CFG.num_layers,
    d_ff=CFG.d_ff,
    max_seq_length=CFG.max_seq_length,
    dropout=0.0,
).to(device)

checkpoint_path = os.path.join(
    CFG.output_dir,
    "virgo_instruction_best.pt"
)

print("Loading:", checkpoint_path)

checkpoint = torch.load(
    checkpoint_path,
    map_location=device,
)

if isinstance(checkpoint, dict) and "model_state_dict" in checkpoint:
    model.load_state_dict(checkpoint["model_state_dict"])
else:
    model.load_state_dict(checkpoint)

model.eval()

print("✅ Virgo loaded successfully!")

# ==========================================================
# Typewriter
# ==========================================================

def typewriter(text, delay=0.015):

    for ch in text:
        print(ch, end="", flush=True)
        time.sleep(delay)

    print()

# ==========================================================
# Generate
# ==========================================================

def generate(
    user_prompt,
    max_new_tokens=256,
    temperature=0.8,
    top_k=40,
    top_p=0.95,
    repetition_penalty=1.1,
    stream=True,
):

    prompt = (
        "<bos>"
        f"User: {user_prompt}"
        "<newline><newline>"
        "Assistant:"
    )

    encoding = tokenizer.encode(prompt)

    input_ids = torch.tensor(
        [encoding.ids],
        dtype=torch.long,
        device=device,
    )

    eos_id = tokenizer.token_to_id("<eos>")

    prompt_length = input_ids.size(1)

    with torch.inference_mode():

        for _ in range(max_new_tokens):

            x = input_ids[:, -CFG.max_seq_length:]

            logits = model(x)

            logits = logits[:, -1, :]

            # repetition penalty
            if repetition_penalty != 1.0:

                for token in set(input_ids[0].tolist()):

                    logits[0, token] /= repetition_penalty

            temperature = max(temperature, 1e-5)

            logits = logits / temperature

            # ---------------- Top-k ----------------

            if top_k > 0:

                values, indices = torch.topk(logits, top_k)

                filtered = torch.full_like(
                    logits,
                    float("-inf")
                )

                filtered.scatter_(1, indices, values)

                logits = filtered

            probs = torch.softmax(logits, dim=-1)

            # ---------------- Top-p ----------------

            if top_p < 1.0:

                sorted_probs, sorted_indices = torch.sort(
                    probs,
                    descending=True
                )

                cumulative = torch.cumsum(
                    sorted_probs,
                    dim=-1
                )

                remove = cumulative > top_p

                remove[...,1:] = remove[..., :-1].clone()

                remove[...,0] = False

                sorted_probs[remove] = 0

                sorted_probs /= sorted_probs.sum(
                    dim=-1,
                    keepdim=True
                )

                sampled = torch.multinomial(
                    sorted_probs,
                    1
                )

                next_token = sorted_indices.gather(
                    1,
                    sampled
                )

            else:

                next_token = torch.multinomial(
                    probs,
                    1
                )

            input_ids = torch.cat(
                (input_ids, next_token),
                dim=1
            )

            if eos_id is not None:

                if next_token.item() == eos_id:
                    break

    generated_ids = input_ids[0][prompt_length:].tolist()

    response = tokenizer.decode(generated_ids)

    for token in [
        "<bos>",
        "<eos>",
        "<pad>",
    ]:
        response = response.replace(token, "")

    response = response.replace("<newline>", "\n")

    response = response.strip()

    if stream:

        typewriter(response)

    else:

        return response

# ==========================================================
# Chat
# ==========================================================

print("="*60)
print("Virgo Chat")
print("Type 'exit' to quit.")
print("="*60)

while True:

    user = input("\n🧑 You: ").strip()

    if user.lower() in [
        "exit",
        "quit",
        "bye",
    ]:
        print("\n👋 Goodbye!")
        break

    print("\n🤖 Virgo: ", end="", flush=True)

    generate(
        user_prompt=user,
        max_new_tokens=256,
        temperature=0.7,
        top_k=40,
        top_p=0.9,
        repetition_penalty=1.15,
        stream=True,
    )

Using: cuda
Loading: /kaggle/working/virgo_chat_v2/virgo_instruction_best.pt
✅ Virgo loaded successfully!
Virgo Chat
Type 'exit' to quit.



🧑 You:  hi



🤖 Virgo: hiismsismsismsismsismsismsismsismsismsismsismsismsismsismsismsismsismsismsismsismsismsismsismsismsismsismsismsismsismsismsismsismsismsismsismsismsismsismsismsismsismsismsismsismsismsismsismsismsismsismsismsismsismsismsismsisms.Virgo:



🧑 You:  introduce yourself



🤖 Virgo: onedededededededededededededededededededededededededededededededededededededededededededededededededededededededededededededededededededed
